## Векторные представления слов (word embeddings)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import random
import numpy as np
from sklearn.feature_extraction import text
from sklearn.model_selection import train_test_split
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
random.seed(1228)
pd.set_option('display.max_colwidth', None)

%matplotlib inline

# Анализ тональности Твиттера

Сегодня мы будем решать задачу анализа тональности для Твиттера.
Наша задача - для каждого Твита по тексту предсказать, положительный он или отрицательный.

Скачиваем куски датасета твитов ([источник](http://study.mokoron.com/)): [положительные](https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv?dl=0), [отрицательные](https://www.dropbox.com/s/ilkte35m35l38mr/negative.sql).

In [2]:
# !unzip twitter.zip

In [6]:
from pymystem3 import Mystem
import re


m = Mystem()


regex = re.compile(r"[А-ЯЁёа-я:=!\)\()A-z\_\%/|]+")

def words_only(text, regex=regex):
    try:
        return " ".join(regex.findall(text))
    except:
        return ""



def lemmatize(text, mystem=m):
    try:
        return "".join(m.lemmatize(text)).strip()
    except:
        return " "
words_only('g;iuhoikl 7.kjh 87h одлжд :))')

'g iuhoikl kjh h одлжд :))'

In [7]:
df_pos = pd.read_csv("./twitter/positive.csv", sep=';', header = None, usecols = [3])
df_pos.tail()

,3
114906,"Спала в родительском доме, на своей кровати... Проснулась с кошкой на голове))"
114907,"RT @jebesilofyt: Эх... Мы немного решили сократить путь, сейчас уже лежу в мягкой кровати :) а с отсутствием сети помогла справится какая-"
114908,"Что происходит со мной, когда в эфире #proactivefm звучит моя любимая песня)) #dctalk #music @… http://t.co/65KGFFd5oO"
114909,"""Любимая,я подарю тебе эту звезду..."" Имя какой звезды переводится ""подмышка""? ;-)"
114910,@Ma_che_rie посмотри #непытайтесьпокинутьомск сегодня в Вавилоне в 18.20. Я там тоже есть :)


In [8]:
df_pos = pd.read_csv("./twitter/negative.csv", sep=';', header = None, usecols = [3])
df_pos.head()

,3
0,"на работе был полный пиддес :| и так каждое закрытие месяца, я же свихнусь так D:"
1,"Коллеги сидят рубятся в Urban terror, а я из-за долбанной винды не могу :("
2,@elina_4post как говорят обещаного три года ждут...((
3,"Желаю хорошего полёта и удачной посадки,я буду очень сильно скучать( http://t.co/jCLNzVNv3S"
4,"Обновил за каким-то лешим surf, теперь не работает простоплеер :("


**Вопрос 1:** Стоит ли удалять пунктуацию на этапе предобработки? Если нет, то почему?

**Вопрос 2:** Лемматизация или стемминг, какой метод лучше выбрать?

In [9]:
df_neg = pd.read_csv("./twitter/negative.csv", sep=';', header = None, usecols = [3])
df_pos = pd.read_csv("./twitter/positive.csv", sep=';', header = None, usecols = [3])
df_neg['sent'] = 'neg'
df_pos['sent'] = 'pos'
df_pos['text'] = df_pos[3]
df_neg['text'] = df_neg[3]
df = pd.concat([df_neg, df_pos])
df = df[['text', 'sent']]
%time df.text = df.text.apply(words_only)
#%time df.text = df.text.apply(lemmatize)

CPU times: user 453 ms, sys: 13.7 ms, total: 466 ms
Wall time: 476 ms


Загрузим предобработанные данные. Слова в твитах были лемматизированы и очищены от лишней пунктуации.

In [10]:
# Data load for Windows users
df = pd.read_csv('./twitter/processed_text.csv', index_col = 0)
df.head()

,text,sent
0,на работа быть полный пиддеса :| и так каждый закрытие месяц я же свихиваться так D:,neg
1,коллега сидеть рубиться в Urban terror а я из за долбать винд не мочь :(,neg
2,elina_ post как говорить обещаной три год ждать ((,neg
3,желать хороший пол тот и удачный посадка я быть очень сильно скучать( http://t co/jCLNzVNv S,neg
4,обновлять за какой то леший surf теперь не работать простоплеер :(,neg


In [11]:
df.shape
#df.head()

(226834, 2)

In [12]:
texts = [df.text.iloc[i].split() for i in range(len(df))]
texts[0]

['на',
 'работа',
 'быть',
 'полный',
 'пиддеса',
 ':|',
 'и',
 'так',
 'каждый',
 'закрытие',
 'месяц',
 'я',
 'же',
 'свихиваться',
 'так',
 'D:']

## Word2Vec модель

Алгоритм Word2Vec реализован в библиотеке `gensim`. Обучение модели - это всего несколько строк кода.



In [21]:
from gensim.models import Word2Vec

In [22]:
model = Word2Vec(texts, vector_size=300, window=5, min_count=5, workers=4)
model.save("word2v.model")

In [23]:
#model = Word2Vec.load("word2v.model")

Посмотрим на близкие слова.

In [24]:
model.wv.most_similar("тоска")

[('грусть', 0.8344243168830872),
 ('заливаться', 0.8245692849159241),
 ('навертываться', 0.8074848651885986),
 ('co/rN', 0.8025125861167908),
 ('адский', 0.7983716726303101),
 ('глазик', 0.7967483401298523),
 ('жор', 0.7960219383239746),
 ('реконнект', 0.791882336139679),
 ('лезвие', 0.7908430695533752),
 ('скупой', 0.7876051664352417)]

In [25]:
model.wv.most_similar("работа")

[('учеба', 0.7485488653182983),
 ('обед', 0.7019277215003967),
 ('опаздывать', 0.6915215253829956),
 ('часы', 0.6701469421386719),
 ('машина', 0.6634623408317566),
 ('тренировка', 0.6634198427200317),
 ('каток', 0.6502974033355713),
 ('завтрак', 0.6460518836975098),
 ('дорога', 0.6422540545463562),
 ('работа(', 0.6409292817115784)]

In [26]:
model.wv.most_similar("отпуск")

[('сентябрь', 0.8259491920471191),
 ('перерыв', 0.7968906164169312),
 ('поездка', 0.7949238419532776),
 ('март', 0.7874515652656555),
 ('чемодан', 0.7850503921508789),
 ('отдых', 0.7829638123512268),
 ('январь', 0.7799921035766602),
 ('февраль', 0.7698257565498352),
 ('июль', 0.7639325261116028),
 ('гулянка', 0.7614350318908691)]

In [27]:
vec = (model.wv['университет'] - model.wv['студент'] + model.wv['школьник'])/3
model.wv.similar_by_vector(vec)

[('поступление', 0.8495321869850159),
 ('университет', 0.8488895893096924),
 ('конгениальность!!!', 0.8038240075111389),
 ('мисс', 0.7861528396606445),
 ('тематика', 0.7744033336639404),
 ('viber', 0.7713941335678101),
 ('палата', 0.7639152407646179),
 ('How', 0.7613934874534607),
 ('блок', 0.7609426379203796),
 ('запуск', 0.760564386844635)]

In [28]:
vec = (model.wv['лето'] - model.wv['жара'] + model.wv['холод'])/3
model.wv.similar_by_vector(vec)

[('лето', 0.9498748779296875),
 ('весна', 0.8095774054527283),
 ('каникулы', 0.7500930428504944),
 ('зима', 0.7342706918716431),
 ('осень', 0.722480833530426),
 ('выходной', 0.679669976234436),
 ('праздник', 0.6635429859161377),
 ('холод', 0.6525872945785522),
 ('поездка', 0.6418941020965576),
 ('снег', 0.6361593008041382)]

In [29]:
model.wv.doesnt_match("ночь улица фонарь аптека".split())

'ночь'

In [30]:
model.wv.doesnt_match("цветок дерево кактус еда".split())

'еда'

In [31]:
model.wv.doesnt_match("рука нога камень".split())

'камень'

### Визуализация пространства слов

Возьмем 500 наиболее частотных слов и визуализируем их векторные представления на плоскости с помощью TSNE.

In [43]:
top_words = []
from nltk import FreqDist
fd = FreqDist()
for text in texts:
    fd.update(text)
for i in fd.most_common(500):
    top_words.append(i[0])
print(top_words)

['я', 'не', 'и', 'в', 'на', 'что', 'RT', 'а', 'http://t', 'быть', 'ты', 'с', 'как', 'то', 'у', 'это', ':(', 'так', ':', ')', '(', 'но', 'все', 'он', 'ну', 'по', 'мы', ':)', 'мой', 'за', 'весь', 'хотеть', 'такой', ':D', 'уже', 'этот', 'вот', 'же', 'только', 'день', 'да', 'кто', 'еще', 'сегодня', 'она', 'бы', 'когда', 'мочь', 'они', 'вы', 'к', 'очень', 'из', 'просто', 'нет', 'один', 'знать', 'какой', 'от', 'год', 'если', 'теперь', 'любить', 'человек', 'свой', 'co/', 'даже', 'о', 'надо', 'до', 'новый', 'завтра', 'тоже', 'там', 'вс', 'тот', 'вообще', '((', 'для', 'самый', 'хороший', 'себя', 'хорошо', 'сейчас', '))', 'почему', 'делать', 'который', 'думать', 'много', 'раз', 'понимать', 'сказать', '(((', 'блин', 'смотреть', 'без', 'время', 'спать', 'спасибо', 'или', 'тут', 'утро', ')))', 'говорить', 'идти', 'сидеть', 'со', 'пойти', 'давать', 'про', 'ничто', 'писать', 'сделать', 'всегда', 'можно', 'жизнь', 'друг', 'первый', 'скоро', 'сам', 'наш', 'где', 'мама', 'потом', 'школа', '!', 'час', 'т

In [44]:
top_words_vec = model.wv[top_words]

In [45]:
top_words_vec.shape

(500, 300)

In [46]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=0)
top_words_tsne = tsne.fit_transform(top_words_vec)

In [48]:
from bokeh.models import ColumnDataSource, LabelSet
from bokeh.plotting import figure, show, output_file
from bokeh.io import output_notebook
output_notebook()

p = figure(tools="pan,wheel_zoom,reset,save",
           toolbar_location="above",
           title="word2vec T-SNE for most common words")

source = ColumnDataSource(data=dict(x1=top_words_tsne[:,0],
                                    x2=top_words_tsne[:,1],
                                    names=top_words))

p.scatter(x="x1", y="x2", size=8, source=source)

labels = LabelSet(x="x1", y="x2", text="names", y_offset=6,
                  text_font_size="8pt", text_color="#555555",
                  source=source, text_align='center')
p.add_layout(labels)

show(p)

Loading BokehJS ...

### Кластеризация слов

In [49]:
from sklearn.metrics.pairwise import cosine_similarity
dist = 1 - cosine_similarity(top_words_vec)

In [67]:
from scipy.cluster.hierarchy import  ward, dendrogram

linkage_matrix = ward(dist)

# fig, ax = plt.subplots(figsize=(10, 100))
# ax = dendrogram(linkage_matrix, orientation="right", labels=top_words);

# plt.tick_params(\
#     axis= 'x',          # changes apply to the x-axis
#     which='both',      # both major and minor ticks are affected
#     bottom='off',      # ticks along the bottom edge are off
#     top='off',         # ticks along the top edge are off
#     labelbottom='off')

# plt.tight_layout()

# plt.savefig('w2v_cluster.png', dpi=200) #save figure as ward_clusters

## Классификация текстов
По мотивам [поста](http://nadbordrozd.github.io/blog/2016/05/20/text-classification-with-word2vec/)


Векторные представления получены, но для решения исходной задачи необходимо получить векторные представления для предложений. Для этого усредним векторы входящих в предложения слов. Тем самым, вектор предложения получится как средний вектор входящих в него слов.

In [51]:
X = df.text.tolist()
y = df.sent.tolist()

X, y = np.array(X), np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.33, random_state = 1)
print ("total train examples %s" % len(y_train))
print ("total test examples %s" % len(y_test))

total train examples 151978
total test examples 74856


## Mean Vector

In [52]:
class MeanEmbeddingVectorizer(object):
    def __init__(self, word2vec):
        self.word2vec = word2vec
        # if a text is empty we should return a vector of zeros
        # with the same dimensionality as all the other vectors
        self.dim = len(w2v.popitem()[1])

    def fit(self, X, y):
        return self

    def transform(self, X):
        return np.array([
            np.mean([self.word2vec[w] for w in words if w in self.word2vec]
                    or [np.zeros(self.dim)], axis=0)
            for words in X
        ])



In [53]:
w2v = dict(zip(model.wv.index_to_key, model.wv.vectors))

In [54]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rfc_w2v = Pipeline([
    ("word2vec vectorizer", MeanEmbeddingVectorizer(w2v)),
    ("extra trees", RandomForestClassifier(n_estimators=20))])


In [55]:
rfc_w2v.fit(X_train,y_train)
pred = rfc_w2v.predict(X_test)

In [56]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

In [57]:
print("Precision: {0:6.2f}".format(precision_score(y_test, pred, average='macro')))
print("Recall: {0:6.2f}".format(recall_score(y_test, pred, average='macro')))
print("F1-measure: {0:6.2f}".format(f1_score(y_test, pred, average='macro')))
print("Accuracy: {0:6.2f}".format(accuracy_score(y_test, pred)))
print(classification_report(y_test, pred))
labels = rfc_w2v.classes_

classification_report
#sns.heatmap(data=confusion_matrix(y_test, pred), annot=True, fmt="d", cbar=False, xticklabels=labels, yticklabels=labels)
#plt.title("Confusion matrix")
#plt.show()

Precision:   0.81
Recall:   0.81
F1-measure:   0.81
Accuracy:   0.81
              precision    recall  f1-score   support

         neg       0.80      0.83      0.81     37150
         pos       0.83      0.79      0.81     37706

    accuracy                           0.81     74856
   macro avg       0.81      0.81      0.81     74856
weighted avg       0.81      0.81      0.81     74856



<function sklearn.metrics._classification.classification_report(y_true, y_pred, *, labels=None, target_names=None, sample_weight=None, digits=2, output_dict=False, zero_division='warn')>

# Средний вектор с весами tf-idf

Теперь воспользуемся более продвинутым методом получения эмбеддингов для предложений. Усредним вектора слов в предложении с весами Tf-Idf.


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [62]:
class TfidfEmbeddingVectorizer(object):
    def __init__(self, word2vec):
        self.word2vec = word2vec
        self.word2weight = None
        self.dim = len(w2v.popitem()[1])

    def fit(self, X, y):
        tfidf = TfidfVectorizer(analyzer=lambda x: x)
        tfidf.fit(X)
        max_idf = max(tfidf.idf_)
        self.word2weight = defaultdict(
            lambda: max_idf,
            [(w, tfidf.idf_[i]) for w, i in tfidf.vocabulary_.items()])

        return self

    def transform(self, X):
        return np.array([
                np.mean([self.word2vec[w] * self.word2weight[w]
                         for w in words if w in self.word2vec] or
                        [np.zeros(self.dim)], axis=0)
                for words in X
            ])

# Классификации

Эврика! Теперь каждый Твит представлен в виде вектора из 300 компонент. К таким данным мы можем применить любой ML алгоритм классификации. Например, RandomForest.

In [63]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rfc_w2v = Pipeline([
    ("word2vec vectorizer", TfidfEmbeddingVectorizer(w2v)),
    ("extra trees", RandomForestClassifier(n_estimators=20))])


In [64]:
rfc_w2v.fit(X_train,y_train)
pred = rfc_w2v.predict(X_test)

In [65]:
print("Precision: {0:6.2f}".format(precision_score(y_test, pred, average='macro')))
print("Recall: {0:6.2f}".format(recall_score(y_test, pred, average='macro')))
print("F1-measure: {0:6.2f}".format(f1_score(y_test, pred, average='macro')))
print("Accuracy: {0:6.2f}".format(accuracy_score(y_test, pred)))
print(classification_report(y_test, pred))
labels = rfc_w2v.classes_

classification_report
#sns.heatmap(data=confusion_matrix(y_test, pred), annot=True, fmt="d", cbar=False, xticklabels=labels, yticklabels=labels)
#plt.title("Confusion matrix")
#plt.show()

Precision:   0.86
Recall:   0.86
F1-measure:   0.86
Accuracy:   0.86
              precision    recall  f1-score   support

         neg       0.85      0.87      0.86     37150
         pos       0.87      0.84      0.86     37706

    accuracy                           0.86     74856
   macro avg       0.86      0.86      0.86     74856
weighted avg       0.86      0.86      0.86     74856



<function sklearn.metrics._classification.classification_report(y_true, y_pred, *, labels=None, target_names=None, sample_weight=None, digits=2, output_dict=False, zero_division='warn')>

# Бонусное задание

 1) зайдите на сайт RusVectores и скачайте одну из предобученных моделей gensim: https://rusvectores.org/ru/models/

2) Проведите аналогичные эксперименты с использованием скачанных векторных представлений

Как изменился результат?